# 02 - QAOA Execution

This notebook demonstrates how to implement and execute the Quantum Approximate Optimization Algorithm (QAOA) for Max-Cut.

## Table of Contents
1. [QAOA Circuit Overview](#overview)
2. [Building QAOA Circuits](#circuit)
3. [Classical Optimization Loop](#optimization)
4. [Execution with Qiskit](#execution)
5. [Energy Landscape Analysis](#landscape)

<a id='overview'></a>
## 1. QAOA Circuit Overview

### Algorithm Overview

QAOA combines:
1. **Quantum Circuit**: Prepares parameterized quantum state
2. **Classical Optimizer**: Finds optimal parameters

### Circuit Structure

For $p$ layers:

```
|ψ⟩ = U_M(β_p) U_C(γ_p) ... U_M(β_1) U_C(γ_1) |+⟩⊗n
```

Where:
- $U_C(\gamma) = e^{-i\gamma H_C}$: Cost Hamiltonian unitary
- $U_M(\beta) = e^{-i\beta H_M}$: Mixer Hamiltonian unitary
- $|+⟩ = (|0⟩ + |1⟩)/\sqrt{2}$: Initial superposition

<a id='circuit'></a>
## 2. Building QAOA Circuits

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

from src.graph_generator import GraphGenerator
from src.hamiltonian_builder import HamiltonianBuilder
from src.qaoa_circuit import QAOACircuitBuilder

# Generate test graph
gen = GraphGenerator(seed=42)
graph = gen.generate_d_regular_graph(n_nodes=8, degree=3, seed=42)

print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [ ]:
# Build Hamiltonian
builder = HamiltonianBuilder()
hamiltonian, offset = builder.build_maxcut_hamiltonian(graph)

print(f"Hamiltonian: {len(hamiltonian)} terms")

# Create QAOA circuit builder
n_qubits = graph.number_of_nodes()
p = 2  # Number of layers

circuit_builder = QAOACircuitBuilder(n_qubits=n_qubits, p=p)

# Build the circuit
qaoa_circuit = circuit_builder.build_qaoa_circuit(hamiltonian)

print(f"\nQAOA Circuit:")
print(f"  Qubits: {qaoa_circuit.num_qubits}")
print(f"  Depth: {qaoa_circuit.depth()}")
print(f"  Parameters: {qaoa_circuit.num_parameters}")

In [ ]:
# Visualize the circuit
plt.figure(figsize=(14, 6))
qaoa_circuit.draw(output='mpl', style='iqp', fold=40)
plt.title(f'QAOA Circuit (p={p} layers, {n_qubits} qubits)')
plt.show()

### Circuit Explanation

The circuit consists of:
1. **H gates**: Initialize to superposition $|+⟩$ state
2. **RZZ gates**: Implement cost unitary $e^{-i\gamma Z_i Z_j}$
3. **RX gates**: Implement mixer unitary $e^{-i\beta X_i}$

<a id='optimization'></a>
## 3. Classical Optimization Loop

In [ ]:
from qiskit_aer import AerSimulator
from src.qaoa_optimizer import QAOAOptimizer

# Create a simple objective function using statevector
def objective_function(params):
    """
    Compute expectation value of Hamiltonian for given parameters.
    """
    # Create circuit with parameters
    gamma = params[0]
    beta = params[1]
    
    # Build simple QAOA circuit
    qc = QuantumCircuit(n_qubits)
    
    # Initialize
    for i in range(n_qubits):
        qc.h(i)
    
    # Cost layer
    for i, j in graph.edges():
        qc.rzz(2 * gamma, i, j)
    
    # Mixer layer
    for i in range(n_qubits):
        qc.rx(2 * beta, i)
    
    # Simulate and get statevector
    simulator = AerSimulator()
    job = simulator.run(qc)
    result = job.result()
    statevector = result.get_statevector(qc)
    
    # Compute expectation value without full matrix
    from qiskit.quantum_info import Statevector
    sv = Statevector(statevector)
    exp_value = sv.expectation_value(hamiltonian).real
    
    return exp_value

print("Objective function defined")

In [ ]:
# Run optimization
optimizer = QAOAOptimizer(
    p=1,
    optimizer_type='COBYLA',
    maxiter=100,
    seed=42
)

# Run optimization
result = optimizer.optimize(
    objective_function=objective_function,
    n_qubits=n_qubits,
    graph=graph
)

print(f"\nOptimization Results:")
print(f"  Optimal parameters: γ={result.optimal_params[0]:.4f}, β={result.optimal_params[1]:.4f}")
print(f"  Optimal value: {result.optimal_value:.4f}")
print(f"  Function evaluations: {result.n_evaluations}")
print(f"  Converged: {result.converged}")

In [ ]:
# Plot convergence
if result.history:
    from src.visualization import Visualizer
    
    viz = Visualizer()
    
    iterations, values = optimizer.get_convergence_plot_data(result)
    
    plt.figure(figsize=(10, 5))
    plt.plot(iterations, values, 'o-', color='orange')
    plt.xlabel('Iteration')
    plt.ylabel('Cost Function Value')
    plt.title('QAOA Optimization Convergence')
    plt.grid(True, alpha=0.3)
    plt.show()

<a id='execution'></a>
## 4. Execution with Qiskit Runtime

In [ ]:
# Execute using Aer simulator
from src.runtime_executor import RuntimeExecutor

# Create executor
executor = RuntimeExecutor(
    mode='local',
    shots=1024
)

# Get backend info
backend_info = executor.get_backend_info()
print(f"Backend: {backend_info}")

In [ ]:
# Create and run circuit with optimal parameters
gamma_opt = result.optimal_params[0]
beta_opt = result.optimal_params[1]

# Build circuit
qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.h(i)

# Cost layer
for i, j in graph.edges():
    qc.rzz(2 * gamma_opt, i, j)

# Mixer layer
for i in range(n_qubits):
    qc.rx(2 * beta_opt, i)

# Run
exec_result = executor.execute_circuit(qc, hamiltonian)

print(f"Execution Result:")
print(f"  Expectation value: {exec_result.expectation_value:.4f}")
print(f"  Circuit depth: {exec_result.circuit_depth}")

<a id='landscape'></a>
## 5. Energy Landscape Analysis

In [ ]:
# Compute energy landscape
gamma_range = np.linspace(0, np.pi, 30)
beta_range = np.linspace(0, np.pi, 30)

gamma_grid, beta_grid = np.meshgrid(gamma_range, beta_range)
cost_grid = np.zeros_like(gamma_grid)

print("Computing energy landscape...")
for i in range(len(gamma_range)):
    for j in range(len(beta_range)):
        params = np.array([gamma_grid[i, j], beta_grid[i, j]])
        cost_grid[i, j] = objective_function(params)
    
    if (i + 1) % 10 == 0:
        print(f"  Completed {i+1}/{len(gamma_range)} rows")

print("Energy landscape computed!")

In [ ]:
# Plot energy landscape
plt.figure(figsize=(10, 8))

contour = plt.contourf(gamma_grid, beta_grid, cost_grid, levels=30, cmap='viridis')
plt.colorbar(contour, label='Cost Function Value')

# Mark optimal point
plt.plot(gamma_opt, beta_opt, 'r*', markersize=15, label=f'Optimal: γ={gamma_opt:.3f}, β={beta_opt:.3f}')

plt.xlabel(r'$\gamma$ (gamma)')
plt.ylabel(r'$\beta$ (beta)')
plt.title('QAOA Energy Landscape')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

In this notebook, we covered:

1. **QAOA Circuit Structure**: How to build parameterized quantum circuits
2. **Cost Hamiltonian**: RZZ gates implement $e^{-i\gamma Z_i Z_j}$
3. **Mixer Hamiltonian**: RX gates implement $e^{-i\beta X_i}$
4. **Classical Optimization**: Using COBYLA to find optimal parameters
5. **Energy Landscape**: Visualizing the cost function over parameter space

In the next notebook, we'll analyze results and compare with classical solutions!